In [1]:
!pip install -q handyspark findspark

In [1]:
import os
import numpy as np
if not hasattr(np, "bool"):
    np.bool = np.bool_   # redirect to the correct scalar type
import findspark
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from handyspark import *
from matplotlib import pyplot as plt
# fixes issue with seaborn hiding fliers on boxplot
import matplotlib as mpl
mpl.rc("lines", markeredgewidth=0.5)

findspark.init()
os.environ['PYSPARK_SUBMIT_ARGS'] = '--master local[*] pyspark-shell'

spark = SparkSession.builder \
    .appName("InstaCartEDA").enableHiveSupport().getOrCreate()

%matplotlib inline

/tmp/ipykernel_7231/287089385.py:3: FutureWarning: In the future `np.bool` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "bool"):


In [2]:
aisles = spark.read.csv("hdfs://hadoop-namenode:9000/user/project/data/raw/aisles.csv", header=True, inferSchema=True)
departments = spark.read.csv("hdfs://hadoop-namenode:9000/user/project/data/raw/departments.csv", header=True, inferSchema=True)
products = spark.read.csv("hdfs://hadoop-namenode:9000/user/project/data/raw/products.csv", header=True, inferSchema=True)
orders = spark.read.csv("hdfs://hadoop-namenode:9000/user/project/data/raw/orders.csv", header=True, inferSchema=True)
order_products_prior = spark.read.csv("hdfs://hadoop-namenode:9000/user/project/data/raw/order_products__prior.csv", header=True, inferSchema=True)
order_products_train = spark.read.csv("hdfs://hadoop-namenode:9000/user/project/data/raw/order_products__train.csv", header=True, inferSchema=True)

In [3]:
def null_report(df):
    """
    Returns a Spark DataFrame with null counts per column.
    Includes: true nulls, empty strings, 'NULL', 'None', and NaN.
    """
    return df.select([
        F.count(
            F.when(
                F.col(c).isNull() | 
                (F.col(c) == '') | 
                (F.col(c) == 'NULL') | 
                (F.col(c) == 'None') | 
                F.isnan(F.col(c)), 
                c
            )
        ).alias(c)
        for c in df.columns
    ])

In [4]:
dfs_dict = {
    "aisles": aisles,
    "departments": departments,
    "products": products,
    "orders": orders,
    "order_products_prior": order_products_prior,
    "order_products_train": order_products_train
}
for name, df in dfs_dict.items():
    print(f'Shape of the dataset: {name}', (df.count(), len(df.columns)))
    df.show(n=10)
    df.printSchema()

Shape of the dataset: aisles (134, 2)
+--------+--------------------+
|aisle_id|               aisle|
+--------+--------------------+
|       1|prepared soups sa...|
|       2|   specialty cheeses|
|       3| energy granola bars|
|       4|       instant foods|
|       5|marinades meat pr...|
|       6|               other|
|       7|       packaged meat|
|       8|     bakery desserts|
|       9|         pasta sauce|
|      10|    kitchen supplies|
+--------+--------------------+
only showing top 10 rows

root
 |-- aisle_id: integer (nullable = true)
 |-- aisle: string (nullable = true)

Shape of the dataset: departments (21, 2)
+-------------+---------------+
|department_id|     department|
+-------------+---------------+
|            1|         frozen|
|            2|          other|
|            3|         bakery|
|            4|        produce|
|            5|        alcohol|
|            6|  international|
|            7|      beverages|
|            8|           pets|
|         

### First Problem Found: 
#### aisle_id, department_id are both string in products

In [5]:
# Null Exploration
print("Looking for Null Values ...")
for name, df in dfs_dict.items():
    print(f"Table: {name}")
    null_report(df).show()

Looking for Null Values ...
Table: aisles
+--------+-----+
|aisle_id|aisle|
+--------+-----+
|       0|    0|
+--------+-----+

Table: departments
+-------------+----------+
|department_id|department|
+-------------+----------+
|            0|         0|
+-------------+----------+

Table: products
+----------+------------+--------+-------------+
|product_id|product_name|aisle_id|department_id|
+----------+------------+--------+-------------+
|         0|           0|       0|            0|
+----------+------------+--------+-------------+

Table: orders
+--------+-------+--------+------------+---------+-----------------+----------------------+
|order_id|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|
+--------+-------+--------+------------+---------+-----------------+----------------------+
|       0|      0|       0|           0|        0|                0|                206209|
+--------+-------+--------+------------+---------+-----------------+-----

### Second Problem Found:
#### Null Values found in 'orders', col: 'days_since_prior_order'

In [6]:
for title, df in dfs_dict.items():
    print(f"Table: {title}")
    df.describe().show()


Table: aisles
+-------+-----------------+--------------------+
|summary|         aisle_id|               aisle|
+-------+-----------------+--------------------+
|  count|              134|                 134|
|   mean|             67.5|                NULL|
| stddev|38.82653731663435|                NULL|
|    min|                1|air fresheners ca...|
|    max|              134|              yogurt|
+-------+-----------------+--------------------+

Table: departments
+-------+------------------+----------+
|summary|     department_id|department|
+-------+------------------+----------+
|  count|                21|        21|
|   mean|              11.0|      NULL|
| stddev|6.2048368229954285|      NULL|
|    min|                 1|   alcohol|
|    max|                21|    snacks|
+-------+------------------+----------+

Table: products
+-------+------------------+--------------------+-----------------+------------------+
|summary|        product_id|        product_name|         ais

## Fixing Problems:
#### - Data Integrity in 'products'
#### - Null Values in 'orders'

In [7]:
products.select("department_id").distinct().show()
products.select("aisle_id").distinct().show()

+-------------+
|department_id|
+-------------+
|            7|
|           15|
|           11|
|            3|
|            8|
|           16|
|            5|
|           18|
|           17|
|         Red"|
|            6|
|           19|
|            9|
|            1|
|           20|
|           10|
|            4|
|           12|
|           13|
|           14|
+-------------+
only showing top 20 rows

+--------+
|aisle_id|
+--------+
|     125|
|       7|
|      51|
|     124|
|      15|
|      54|
|     132|
|      11|
|     101|
|      69|
|      29|
|     112|
|      42|
|      87|
|      73|
|      64|
|       3|
| Blunted|
|      30|
|     113|
+--------+
only showing top 20 rows



In [8]:
products.filter(~products.department_id.cast("int").isNotNull()).show(truncate=False)
products.filter(~products.aisle_id.cast("int").isNotNull()).show(truncate=False)

+----------+-------------------------+--------+-------------+
|product_id|product_name             |aisle_id|department_id|
+----------+-------------------------+--------+-------------+
|6816      |"Scotch Kids 5"" Scissors| Blunted| Red"        |
+----------+-------------------------+--------+-------------+

+----------+-------------------------+--------+-------------+
|product_id|product_name             |aisle_id|department_id|
+----------+-------------------------+--------+-------------+
|6816      |"Scotch Kids 5"" Scissors| Blunted| Red"        |
+----------+-------------------------+--------+-------------+



In [9]:
products = products.filter(~products.product_id.isin(6816))

In [10]:
products.select("department_id").distinct().show()
products.select("aisle_id").distinct().show()

+-------------+
|department_id|
+-------------+
|            7|
|           15|
|           11|
|            3|
|            8|
|           16|
|            5|
|           18|
|           17|
|            6|
|           19|
|            9|
|            1|
|           20|
|           10|
|            4|
|           12|
|           13|
|           14|
|           21|
+-------------+
only showing top 20 rows

+--------+
|aisle_id|
+--------+
|     125|
|       7|
|      51|
|     124|
|      15|
|      54|
|     132|
|      11|
|     101|
|      69|
|      29|
|     112|
|      42|
|      87|
|      73|
|      64|
|       3|
|      30|
|     113|
|      34|
+--------+
only showing top 20 rows



In [11]:
products.columns

['product_id', 'product_name', 'aisle_id', 'department_id']

In [12]:
from pyspark.sql.types import IntegerType
colum = []
for c in products.columns:
    if c in ['department_id', 'aisle_id']:
        colum.append(products[c].cast(IntegerType()).alias(c))
    else:
        colum.append(products[c])
    

In [13]:
products = products.select(colum)
products.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- aisle_id: integer (nullable = true)
 |-- department_id: integer (nullable = true)



First Problem Solved!

In [14]:
orders.columns

['order_id',
 'user_id',
 'eval_set',
 'order_number',
 'order_dow',
 'order_hour_of_day',
 'days_since_prior_order']

In [15]:
orders = orders.na.fill(value=0, subset=["days_since_prior_order"])
orders.select([
    F.count(
        F.when(
            F.col(c).isNull() | 
            (F.col(c) == '') | 
            (F.col(c) == 'NULL') | 
            (F.col(c) == 'None') | 
            F.isnan(F.col(c)), 
            c
        )
    ).alias(c) 
        for c in orders.columns]).show()

+--------+-------+--------+------------+---------+-----------------+----------------------+
|order_id|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|
+--------+-------+--------+------------+---------+-----------------+----------------------+
|       0|      0|       0|           0|        0|                0|                     0|
+--------+-------+--------+------------+---------+-----------------+----------------------+



In [16]:
order_details = (order_products_prior
    .join(products, "product_id")
    .join(aisles, "aisle_id")
    .join(departments, "department_id")
    .join(orders, "order_id"))

In [17]:
print("----- Table: order details -----")
order_details.show(10)
order_details.printSchema()

----- Table: order details -----
+--------+-------------+--------+----------+-----------------+---------+--------------------+--------------------+---------------+-------+--------+------------+---------+-----------------+----------------------+
|order_id|department_id|aisle_id|product_id|add_to_cart_order|reordered|        product_name|               aisle|     department|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|
+--------+-------------+--------+----------+-----------------+---------+--------------------+--------------------+---------------+-------+--------+------------+---------+-----------------+----------------------+
|      31|            7|      94|     48946|                1|        1|       Earl Grey Tea|                 tea|      beverages| 201744|   prior|           7|        6|               15|                  14.0|
|      31|            4|      83|     22935|                2|        1|Organic Yellow Onion|    fresh vegetables|     

In [18]:
hdf = order_details.toHandy()

/usr/local/spark/python/pyspark/sql/dataframe.py:168: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(
/usr/local/spark/python/pyspark/sql/dataframe.py:147: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


In [19]:
total_rows = order_details.count()
null_counts = null_report(order_details).toPandas().T
null_counts.columns = ["null_count"]
null_counts["null_ratio"] = null_counts["null_count"] / total_rows
print(null_counts)

                        null_count  null_ratio
order_id                         0         0.0
department_id                    0         0.0
aisle_id                         0         0.0
product_id                       0         0.0
add_to_cart_order                0         0.0
reordered                        0         0.0
product_name                     0         0.0
aisle                            0         0.0
department                       0         0.0
user_id                          0         0.0
eval_set                         0         0.0
order_number                     0         0.0
order_dow                        0         0.0
order_hour_of_day                0         0.0
days_since_prior_order           0         0.0


In [20]:
# Detect duplicates in order_details
duplicates = (order_details
    .groupBy(order_details.columns)   # group by all columns
    .count()
    .filter("count > 1")              # keep only rows that appear more than once
)
print(f"Number of duplicate rows: {duplicates.count()}")


Number of duplicate rows: 0


In [21]:
from pyspark.sql.functions import col
from pyspark.sql.types import NumericType

def tukey_outliers_all(df, k=1.5):
    """
    Detect Tukey outliers for every numeric column in a DataFrame.
    Returns a dictionary with column name -> outlier count.
    """
    outlier_summary = {}
    numeric_cols = [c for c, t in df.dtypes if isinstance(df.schema[c].dataType, NumericType)]
    
    for c in numeric_cols:
        q1, q3 = df.approxQuantile(c, [0.25, 0.75], 0.01)
        iqr = q3 - q1
        lower, upper = q1 - k * iqr, q3 + k * iqr
        
        # Count outliers
        outlier_count = df.filter((col(c) < lower) | (col(c) > upper)).count()
        outlier_summary[c] = {
            "Q1": q1,
            "Q3": q3,
            "IQR": iqr,
            "LowerBound": lower,
            "UpperBound": upper,
            "OutlierCount": outlier_count
        }
    
    return outlier_summary


In [22]:
summary = tukey_outliers_all(order_details, k=1.5)
for col_name, stats in summary.items():
    print(f"{col_name}: {stats}")

order_id: {'Q1': 842761.0, 'Q3': 2548496.0, 'IQR': 1705735.0, 'LowerBound': -1715841.5, 'UpperBound': 5107098.5, 'OutlierCount': 0}
department_id: {'Q1': 4.0, 'Q3': 16.0, 'IQR': 12.0, 'LowerBound': -14.0, 'UpperBound': 34.0, 'OutlierCount': 0}
aisle_id: {'Q1': 31.0, 'Q3': 107.0, 'IQR': 76.0, 'LowerBound': -83.0, 'UpperBound': 221.0, 'OutlierCount': 0}
product_id: {'Q1': 13649.0, 'Q3': 37766.0, 'IQR': 24117.0, 'LowerBound': -22526.5, 'UpperBound': 73941.5, 'OutlierCount': 0}
add_to_cart_order: {'Q1': 3.0, 'Q3': 12.0, 'IQR': 9.0, 'LowerBound': -10.5, 'UpperBound': 25.5, 'OutlierCount': 997175}
reordered: {'Q1': 0.0, 'Q3': 1.0, 'IQR': 1.0, 'LowerBound': -1.5, 'UpperBound': 2.5, 'OutlierCount': 0}
user_id: {'Q1': 51524.0, 'Q3': 154158.0, 'IQR': 102634.0, 'LowerBound': -102427.0, 'UpperBound': 308109.0, 'OutlierCount': 0}
order_number: {'Q1': 5.0, 'Q3': 24.0, 'IQR': 19.0, 'LowerBound': -23.5, 'UpperBound': 52.5, 'OutlierCount': 1783828}
order_dow: {'Q1': 1.0, 'Q3': 5.0, 'IQR': 4.0, 'LowerBo

In [28]:
order_details.count()

ConnectionRefusedError: [Errno 111] Connection refused

In [27]:
pdf = order_details.sample(fraction=0.01, seed=42).limit(50000).toPandas()

ConnectionRefusedError: [Errno 111] Connection refused

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving


In [30]:
spark.stop()

ConnectionRefusedError: [Errno 111] Connection refused